# Practical 2: Affine transformations and resampling images

## 1. Loading and displaying images

In [1]:
# import the required packages
import numpy as np
import skimage.io
import matplotlib
import matplotlib.pyplot as plt
import utils as ut

# the next lines make figures display in separate windows instead of in the notebook
# this is required for the animations to work correctly
matplotlib.use('TkAgg')
plt.ion()

In [2]:
# Load the 2D lung MRI image using the imread function from the scikit-image python library
img = skimage.io.imread('data/practical2/lung_MRI_slice.png')

# Check the data type of the image
print(img.dtype)

# convert data type to double to avoid errors when processing integers
img = np.double(img)

# check new data type
print(img.dtype)

uint8
float64


In [3]:
# TODO: Add your own code here to reorientate the image into 'standard orientation'
img = img.T
img = np.flip(img, 1)

# display the image using the dispImage function, it should open in a separate window
ut.dispImage(img)

## 2. Translating and resampling images

In [4]:
# Create a 2D array representing a 2D affine matrix for a translation by 10 pixels in x and 20 pixels in y
# Note: numpy has a matrix class, but it recommends not to use it and use standard arrays instead, so arrays should be used in these exercises
# Matrix multiplication can be performed between two arrays using the @ operator or the numpy.matmul function

# TODO: edit the array to represent a 2D affine matrix for a translation by 10 pixels in x and 20 pixels in y
T = np.array([[1, 0, 10], [0, 1, 20], [0, 0, 1]])
print(T)

[[ 1  0 10]
 [ 0  1 20]
 [ 0  0  1]]


In [5]:
# TODO: edit the code below to create a deformation field from the affine matrix
# and then use the deformation field to resample the image
# The function definitions in uitls.py explain what the inputs and outputs of the functions should be
num_pix_x, num_pix_y = img.shape
def_field = ut.defFieldFromAffineMatrix(T, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field)

# display the resampled image
ut.dispImage(img_resampled)

In [7]:
# check the value of pixel 255,255 in the resampled image
print(img_resampled[255,255])

nan


In [6]:
# TODO: add code to resample the image using nearest neighbour and splinef2d interpolation
img_resampled_nn = ut.resampImageWithDefField(img, def_field, interp_method='nearest')
img_resampled_sp = ut.resampImageWithDefField(img, def_field, interp_method='splinef2d')

# TODO: display the results in separate windows
plt.figure()
ut.dispImage(img_resampled_nn)
plt.figure()
ut.dispImage(img_resampled_sp)

In [9]:
# TODO: display the difference images between the new results and the original image in separate windows
plt.figure()
ut.dispImage(img_resampled - img_resampled_nn)
plt.figure()
ut.dispImage(img_resampled - img_resampled_sp)

In [13]:
# TODO: repeat the above steps for a translation by 10.5 pixels in x and 20.5 pixels in y
T = np.array([[1, 0, 10.5], [0, 1, 20.5], [0, 0, 1]])
def_field = ut.defFieldFromAffineMatrix(T, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field)
img_resampled_nn = ut.resampImageWithDefField(img, def_field, interp_method='nearest')
img_resampled_sp = ut.resampImageWithDefField(img, def_field, interp_method='splinef2d')
plt.figure()
ut.dispImage(img_resampled - img_resampled_nn)
plt.figure()
ut.dispImage(img_resampled - img_resampled_sp)

In [14]:
# TODO: display the difference images with intensity limits of [-20, 20]
plt.figure()
ut.dispImage(img_resampled - img_resampled_nn, int_lims=[-20, 20])
plt.figure()
ut.dispImage(img_resampled - img_resampled_sp, int_lims=[-20, 20])

## 3. Rotating images

In [6]:
# define a function to calculate the affine matrix for a rotation about a point
def affineMatrixForRotationAboutPoint(theta, p_coords):
    """
    function to calculate the affine matrix corresponding to an anticlockwise
    rotation about a point

    SYNTAX:
        aff_mat = affineMatrixForRotationAboutPoint(theta, p_coords)
    
    INPUTS:
        theta - the angle of the rotation, specified in degrees
        p_coords - the 2D coordinates of the point that is the centre of rotation.
            p_coords[0] is the x coordinate,
            p_coords[1] is the y coordinate
    
    OUTPUTS:
        aff_mat - a numpy array representing the 3 x 3 affine matrix
    """
    
    # TODO: implement the function
    #convert theta to radians
    theta = np.pi * theta / 180

    #form matrices for translation and rotation
    T1 = np.array([[1, 0, -p_coords[0]],
                  [0, 1, -p_coords[1]],
                  [0,0,1]])
    T2 = np.array([[1, 0, p_coords[0]],
                  [0, 1, p_coords[1]],
                  [0,0,1]])
    R = np.array([[np.cos(theta), -np.sin(theta), 0],
                 [np.sin(theta), np.cos(theta), 0],
                 [0, 0, 1]])
  
    #compose matrices using matrix multiplication
    aff_mat = T2 @ R @ T1

    # return the affine matrix
    return aff_mat

In [7]:
# Close any open figures before continuing
plt.close('all')

# TODO: Use the above function to calculate the affine matrix representing an anticlockwise rotation
# of 5 degrees about the centre of the image
R = affineMatrixForRotationAboutPoint(5, [(num_pix_x - 1)/2, (num_pix_y - 1)/2])
print(R)

# TODO: use this rotation to transform the original image
def_field = ut.defFieldFromAffineMatrix(R, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field)

# TODO: display the resampled image using the intensity limits of the original image
plt.figure()
int_lims_img = [np.min(img), np.max(img)]
ut.dispImage(img_resampled, int_lims=int_lims_img)

[[  0.9961947   -0.08715574  11.59753319]
 [  0.08715574   0.9961947  -10.62718121]
 [  0.           0.           1.        ]]


In [8]:
# TODO: apply the same transformation again to the resampled image and display the result. 
# repeat this 71 times so that the image appears to rotate a full 360 degrees.
for n in range(71):
    img_resampled = ut.resampImageWithDefField(img_resampled, def_field)
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

In [31]:
# TODO: repeat above code with 0 padding value
# need to first resample the original image and display it
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0)
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates
for n in range(71):
    img_resampled = ut.resampImageWithDefField(img_resampled, def_field, pad_value=0)
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

In [32]:
# TODO: repeat above code using nearest neighbour interpolation
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='nearest')
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates
for n in range(71):
    img_resampled = ut.resampImageWithDefField(img_resampled, def_field, pad_value=0, interp_method='nearest')
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

In [34]:
# TODO: repeat above code using splinef2d interpolation
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='splinef2d')
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates
for n in range(71):
    img_resampled = ut.resampImageWithDefField(img_resampled, def_field, pad_value=0, interp_method='splinef2d')
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

## 4. Composing transformations

In [9]:
# TODO: write code below that makes an animation of the rotating image as above (using linear interpolation)
# but composes the transformations to avoid multiple resamplings of the image

import imageio
fig = plt.figure()
frames = []

R_current = R
def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0)
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates

frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
frames.append(frame)

for n in range(71):
    R_current = R_current @ R
    def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
    img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0)
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)

imageio.mimsave('trans_rot_final_image_compose_linear.gif', frames, fps=10, loop=0)

C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\229672114.py:14: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\229672114.py:25: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')


In [ ]:
# TODO: repeat the animation using nearest neighbour and splinef2d interpolation
frames = []

R_current = R
def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='nearest')
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates

frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
frames.append(frame)

for n in range(71):
    R_current = R_current @ R
    def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
    img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='nearest')
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)

imageio.mimsave('trans_rot_final_image_compose_nearest.gif', frames, fps=10, loop=0)


frames = []

R_current = R
def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='splinef2d')
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates

frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
frames.append(frame)

for n in range(71):
    R_current = R_current @ R
    def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
    img_resampled = ut.resampImageWithDefField(img, def_field,  pad_value=0, interp_method='splinef2d')
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)

imageio.mimsave('trans_rot_final_image_compose_splinef2d.gif', frames, fps=10, loop=0)


C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\88696620.py:10: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\88696620.py:21: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\88696620.py:36: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\88696620.py:47: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed 

## 5. Push interpolation

In [ ]:
# TODO: copy and modify the above code to use push interpolation instead of pull interpolation
fig = plt.figure()
frames = []

R_current = R
def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
img_resampled = ut.resampImageWithDefFieldPushInterp(img, def_field)
ut.dispImage(img_resampled, int_lims=int_lims_img)
plt.pause(0.05) # add a short pause so the figure display updates

frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
frames.append(frame)

for n in range(71):
    R_current = R_current @ R
    def_field = ut.defFieldFromAffineMatrix(R_current, num_pix_x, num_pix_y)
    img_resampled = ut.resampImageWithDefFieldPushInterp(img, def_field)
    ut.dispImage(img_resampled, int_lims=int_lims_img)
    plt.pause(0.05) # add a short pause so the figure display updates

    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)

imageio.mimsave('trans_rot_final_image_push.gif', frames, fps=3, loop=0)

C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\3998973965.py:11: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
C:\Users\jmccl\AppData\Local\Temp\ipykernel_17752\3998973965.py:22: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
